In [1]:
from dotenv import load_dotenv
import os
from triplet_extraction.src.db import init_mongo

load_dotenv()
uri = os.getenv("MONGODB_URI")
mongo_client = init_mongo()
if not mongo_client:
    db = None
    print("Failed to connect to MongoDB. Exiting.")
else:
    db = mongo_client["KB_PROPERTY_LAW"]

You successfully connected to MongoDB!


## Extract text from files and save to CSV

In [2]:
from triplet_extraction.src.doc_extraction.pdf_extraction import extract_text_from_pdf
from triplet_extraction.src.doc_extraction.ms_word_extraction import extract_text_from_docx
from triplet_extraction.src.doc_extraction.utils import convert_doc_to_docx
import pandas as pd
from pathlib import Path
import tempfile
import os
from dotenv import load_dotenv
import shutil

load_dotenv()
api_key = os.getenv("LLAMA_INDEX_API_KEY")

data_folder = Path(r"E:\Github\LawAssistant\triplet_extraction\data\luat_dat_dai")
file_folder = data_folder / "files"
output_csv = data_folder / "extracted_texts.csv"

# Create temp folder for doc conversions
temp_conversion_folder = Path(tempfile.mkdtemp(prefix="doc_conversion_"))

# Read the Excel file
df = pd.read_excel(data_folder / "Luat_dat_dai_file.xlsx")

# Load existing extracted data if CSV exists
existing_so_hieu = set()
extracted_data = []

if output_csv.exists():
    existing_df = pd.read_csv(output_csv, encoding='utf-8-sig')
    extracted_data = existing_df.to_dict('records')
    existing_so_hieu = set(existing_df['so_hieu'].tolist())
    print(f"Found existing CSV with {len(existing_so_hieu)} documents")
    print(f"{'='*60}")
else:
    print("No existing CSV found. Starting fresh extraction.")
    print(f"{'='*60}")

# Track statistics
skipped_count = 0
processed_count = 0

for idx, row in df.iterrows():
    so_hieu = row["so_hieu"]
    title = row["title"]
    effective_date = row["effective date"]
    file1 = row["File 1"]
    file2 = row["File 2"]
    file3 = row["File 3"]

    # Skip if already extracted
    if so_hieu in existing_so_hieu:
        skipped_count += 1
        print(f"[{idx+1}/{len(df)}] Skipping {so_hieu} (already extracted)")
        continue

    print(f"\n{'='*60}")
    print(f"[{idx+1}/{len(df)}] Processing: {so_hieu} - {title}")
    print(f"{'='*60}")

    # Collect all file references
    files_to_process = []
    for file_name in [file1, file2, file3]:
        if pd.notna(file_name):
            files_to_process.append(file_name)

    # Combined text from all files for this document
    combined_text = ""
    source_files = []

    # Extract text from all files
    for file_name in files_to_process:
        file_path = file_folder / file_name
        if file_path.exists():
            if file_path.suffix.lower() in [".pdf", ".doc", ".docx"]:
                text = ""

                if file_path.suffix.lower() == ".doc":
                    file_path = convert_doc_to_docx(file_path, temp_conversion_folder)

                print(f"Extracting text from {file_name}...")

                try:
                    if file_path.suffix.lower() == ".docx":
                        text = extract_text_from_docx(file_path)
                    elif file_path.suffix.lower() == ".pdf":
                        text = extract_text_from_pdf(api_key, file_path)

                    if text:
                        combined_text += text
                        source_files.append(file_name)
                        print(f"✓ Extracted {len(text):,} characters from {file_name}")
                    else:
                        print(f"Warning: No text extracted from {file_name}")

                except Exception as e:
                    print(f"Error extracting text from {file_name}: {str(e)}")
                    import traceback
                    traceback.print_exc()

            else:
                print(f"Skipping {file_name}: unsupported format")
        else:
            print(f"Warning: {file_name} not found")

    # Store the extracted data
    if combined_text:
        extracted_data.append({
            "so_hieu": so_hieu,
            "title": title,
            "effective_date": effective_date,
            "source_files": ", ".join(source_files),
            "combined_text": combined_text,
            "text_length": len(combined_text)
        })
        existing_so_hieu.add(so_hieu)  # Add to set to avoid duplicates
        processed_count += 1
        print(f"✓ Total extracted: {len(combined_text):,} characters from {len(source_files)} file(s)")

        # Save incrementally after each successful extraction
        temp_df = pd.DataFrame(extracted_data)
        temp_df.to_csv(output_csv, index=False, encoding='utf-8-sig')
        print(f"✓ Saved progress to CSV")
    else:
        print(f"Warning: No text extracted for {so_hieu}")

# Final save
extracted_df = pd.DataFrame(extracted_data)
extracted_df.to_csv(output_csv, index=False, encoding='utf-8-sig')

print(f"\n{'='*60}")
print(f"EXTRACTION COMPLETE")
print(f"{'='*60}")
print(f"Total documents in CSV: {len(extracted_data)}")
print(f"Documents skipped (already existed): {skipped_count}")
print(f"Documents processed this run: {processed_count}")
print(f"Saved to: {output_csv}")
print(f"{'='*60}")

# Cleanup temp conversion folder
if temp_conversion_folder.exists():
    shutil.rmtree(temp_conversion_folder)
    print(f"Cleaned up temporary conversion folder")

Found existing CSV with 59 documents

[1/63] Processing: 31/2024/QH15 - Luật Đất đai
Extracting text from VanBanGoc_31-2024-qh15_1.pdf...
✓ Extracted 183,049 characters from VanBanGoc_31-2024-qh15_1.pdf
Extracting text from VanBanGoc_31-2024-qh15_2.pdf...


KeyboardInterrupt: 

## Parse sections from extracted texts CSV

In [ ]:
from triplet_extraction.src.doc_extraction.parse_text_to_section import parse_document
import pandas as pd
from pathlib import Path

data_folder = Path(r"E:\Github\LawAssistant\triplet_extraction\data\luat_dat_dai")

# Get the collection
sections_collection = db["legal_sections"]

# Create indexes for better query performance
sections_collection.create_index("so_hieu")
sections_collection.create_index("parent_id")
sections_collection.create_index("full_path", unique=True)

# Read the extracted texts CSV
extracted_csv = data_folder / "extracted_texts.csv"
df = pd.read_csv(extracted_csv, encoding='utf-8-sig')

print(f"Loaded {len(df)} documents from CSV")

total_inserted = 0
total_updated = 0
total_skipped = 0

for idx, row in df.iterrows():
    so_hieu = row["so_hieu"]
    title = row["title"]
    effective_date = row["effective_date"]
    source_files = row["source_files"]
    combined_text = row["combined_text"]

    # Check if document already exists in database
    existing_count = sections_collection.count_documents({"so_hieu": so_hieu})
    if existing_count > 0:
        print(f"\n{'='*60}")
        print(f"[{idx+1}/{len(df)}] SKIPPING: {so_hieu} - {title}")
        print(f"Already in database with {existing_count} sections")
        print(f"{'='*60}")
        total_skipped += 1
        continue

    print(f"\n{'='*60}")
    print(f"[{idx+1}/{len(df)}] Processing: {so_hieu} - {title}")
    print(f"Text length: {len(combined_text):,} characters")
    print(f"{'='*60}")

    try:
        # Parse the combined document
        result = parse_document(combined_text, so_hieu)

        if not result:
            print(f"Warning: No sections parsed from text")
            continue

        print(f"✓ Parsed {len(result)} sections")

        # Insert or update each section in MongoDB
        inserted_count = 0
        updated_count = 0

        for section_id, section_data in result.items():
            # Add metadata
            section_data["document_title"] = title
            section_data["effective_date"] = effective_date
            section_data["source_file"] = source_files

            # Use upsert to insert or update based on full_path
            update_result = sections_collection.update_one(
                {"full_path": section_data["full_path"]},
                {
                    "$set": section_data,
                },
                upsert=True
            )

            if update_result.upserted_id:
                inserted_count += 1
            elif update_result.modified_count > 0:
                updated_count += 1

        print(f"✓ Inserted: {inserted_count} sections")
        print(f"✓ Updated: {updated_count} sections")

        total_inserted += inserted_count
        total_updated += updated_count

    except Exception as e:
        print(f"Error parsing document: {str(e)}")
        import traceback
        traceback.print_exc()

print(f"\n{'='*60}")
print(f"PARSING COMPLETE")
print(f"{'='*60}")
print(f"Documents skipped (already in DB): {total_skipped}")
print(f"Total sections inserted: {total_inserted}")
print(f"Total sections updated: {total_updated}")
print(f"Total sections processed: {total_inserted + total_updated}")
print(f"{'='*60}")

In [6]:
# Add this right after reading the Excel file
df = pd.read_excel(data_folder / "Luat_dat_dai_file.xlsx")

# Print the actual column names
print("Actual columns in Excel file:")
print(df.columns.tolist())
print("\nFirst few rows:")
print(df.head())

Actual columns in Excel file:
[' so_hieu', 'title', 'effective date', 'File 1', 'File 2', 'File 3']

First few rows:
        so_hieu                                              title  \
0  31/2024/QH15                                       Luật Đất đai   
1  91/2015/QH13                                        Luật Dân sự   
2  23/2003/QH11  NGHỊ QUYẾT\n\nVỀ NHÀ ĐẤT DO NHÀ NƯỚC ĐÃ QUẢN L...   
3  43/2024/QH15  LUẬT\nSỬA ĐỔI, BỔ SUNG MỘT SỐ ĐIỀU\nCỦA LUẬT Đ...   
4  59/2020/QH14                                  Luật doanh nghiệp   

        effective date                                     File 1  \
0  2024-01-18 00:00:00               VanBanGoc_31-2024-qh15_1.pdf   
1  2015-11-24 00:00:00              VanBanGoc_91.2015.QH13.P1.pdf   
2  2003-11-26 00:00:00                           23.2003.QH11.doc   
3  2024-06-29 00:00:00  VanBanGoc_2024_987 + 988_43-2024-QH15.pdf   
4  2020-06-17 00:00:00                    VanBanGoc_59.signed.pdf   

                          File 2               